In [0]:
# ============================================================
# EA RT STATIONS — full station and measure register
# Source : environment.data.gov.uk/flood-monitoring/id/stations
# Output : ea_rt_stations (one row per station)
#          ea_rt_station_measures (one row per measure)
# Cadence: manual / on-demand refresh
# ============================================================

import requests
import pandas as pd
import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)
from config import TBL_EA_RT_STATIONS, TBL_EA_RT_STATION_MEASURES

BASE_URL = "https://environment.data.gov.uk/flood-monitoring/id/stations"
LIMIT = 10000

# Paginate through the API until a page returns fewer rows than the limit
all_stations = []
offset = 0

while True:
    response = requests.get(
        BASE_URL,
        params={"_limit": LIMIT, "_offset": offset},
        timeout=120,
    )
    response.raise_for_status()
    page = response.json().get("items", [])
    all_stations.extend(page)
    print(f"  offset={offset}, got {len(page)} stations (total so far: {len(all_stations)})")
    if len(page) < LIMIT:
        break
    offset += LIMIT

print(f"Total stations fetched: {len(all_stations)}")

# Some URI fields return as a list on certain station types — normalise to scalar
def coerce_scalar(val):
    if isinstance(val, list):
        return val[0] if val else None
    return val

# Strip the status URI to just the final path segment
def strip_status(uri):
    if uri and isinstance(uri, str) and "/" in uri:
        return uri.rstrip("/").rsplit("/", 1)[-1]
    return uri

# Extract scalar station fields — every field treated as nullable via .get()
station_rows = []
for s in all_stations:
    station_rows.append({
        "station_id":       coerce_scalar(s.get("@id")),
        "stationReference": coerce_scalar(s.get("stationReference")),
        "notation":         coerce_scalar(s.get("notation")),
        "label":            coerce_scalar(s.get("label")),
        "RLOIid":           coerce_scalar(s.get("RLOIid")),
        "wiskiID":          coerce_scalar(s.get("wiskiID")),
        "catchmentName":    coerce_scalar(s.get("catchmentName")),
        "riverName":        coerce_scalar(s.get("riverName")),
        "town":             coerce_scalar(s.get("town")),
        "dateOpened":       coerce_scalar(s.get("dateOpened")),
        "easting":          coerce_scalar(s.get("easting")),
        "northing":         coerce_scalar(s.get("northing")),
        "lat":              coerce_scalar(s.get("lat")),
        "long":             coerce_scalar(s.get("long")),
        "gridReference":    coerce_scalar(s.get("gridReference")),
        "datumOffset":      coerce_scalar(s.get("datumOffset")),
        "stageScale":       coerce_scalar(s.get("stageScale")),
        "downstageScale":   coerce_scalar(s.get("downstageScale")),
        "status":           strip_status(coerce_scalar(s.get("status"))),
    })

stations_df = pd.DataFrame(station_rows)

# Force identifier columns to string — pandas may infer numeric for fields
# like wiskiID that look like integers, silently dropping leading zeros
for col in ["wiskiID", "stationReference", "notation", "RLOIid", "station_id"]:
    stations_df[col] = stations_df[col].astype(str).where(stations_df[col].notna(), None)

# Explode the measures array — one row per measure, keyed by stationReference
measure_rows = []
for s in all_stations:
    station_ref = s.get("stationReference")
    for m in s.get("measures") or []:
        measure_rows.append({
            "stationReference": station_ref,
            "measure_uri":      m.get("@id"),
            "parameter":        m.get("parameter"),
            "parameterName":    m.get("parameterName"),
            "period":           m.get("period"),
            "qualifier":        m.get("qualifier"),
            "unitName":         m.get("unitName"),
        })

measures_df = pd.DataFrame(measure_rows)

In [0]:
display(stations_df)

In [0]:
#display(measures_df)

In [0]:
# Force identifier columns to string for the same reason
for col in ["stationReference"]:
    measures_df[col] = measures_df[col].astype(str).where(measures_df[col].notna(), None)

# Convert to Spark and write both tables as Delta, overwriting on each run
(
    spark.createDataFrame(stations_df)
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TBL_EA_RT_STATIONS)
)

(
    spark.createDataFrame(measures_df)
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TBL_EA_RT_STATION_MEASURES)
)

# Summary
print(f"Stations written : {len(stations_df):,}")
print(f"Measures written : {len(measures_df):,}")
print("\nMeasures by parameter type:")
print(measures_df["parameter"].value_counts().to_string())